# 潜空间蒸馏与 Prompt 蒸馏

我们继续说基于生成式模型的数据集蒸馏。

本章为你介绍两种将数据集蒸馏迁移到潜空间中的方法，GLaD 与 LD3M，前者使用 VAE Encoder 与 StyleGAN 完成这件事，后者则将 StyleGAN 改为一般 Diffusion 模型。

其次是一种提出将数据集蒸馏形式应用到 Prompt 空间上的方法，D3M。我们假设使用蒸馏数据集的场景天然拥有一套现成 T2I 通用生成模型，这样我们便可以节省非常多压缩空间。这为我们的蒸馏形式打开了新的视野。

# Generative Latent Distillation

原文是 https://arxiv.org/pdf/2305.01649 Generalizing Dataset Distillation via Deep Generative Prior。

GLaD 并不是一种完全独立的方法，而是一种 plug-in 式的方法。我们首先选定一种基本的数据集蒸馏方法，比如 MTT 或 GM。

对于传统基于像素的数据集蒸馏方法，我们会初始化合成数据集像素并且直接对像素做优化。现在我们将初始化像素这一步改为初始化 latent
$$\mathcal Z\sim P_z.$$
其中 $\mathcal Z=\{z_1,\ldots,z_m\}$ 是一组可学习潜空间元素，$P_z$ 是其遵守的初始化分布。

随后我们通过生成器将其生成到原始像素空间中图像
$$\mathcal S=G(\mathcal Z).$$
注意这里生成模型 $G$ 是冻结的。我们对于生成后像素应用我们上述提到的算法，如 MTT 或 GM。
$$\mathcal L=\mathrm{Alg}(\mathcal S,\mathcal T).$$
更新时，我们将梯度传播到原始潜空间像素
$$\mathcal Z
\leftarrow
\operatorname{SGD}
\left(
\mathcal Z,
\nabla_{\mathcal Z}\mathcal L
\right).$$
最后合成数据集合还是
$$\mathcal S=G(\mathcal Z).$$

我们已经说完了，就是如此简短。下面给出完整算法。

$$
\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{Generative Latent Distillation} \\
\hline
\textbf{Input: } \text{Alg: Distillation algorithm (MTT, DC, or DM).} \\
\textbf{Input: } \mathcal{T}\text{: Real training set.} \\
\textbf{Input: } \mathcal{A}\text{: Differentiable augmentation function.} \\
\textbf{Input: } G\text{: Pre-trained generator.} \\
\textbf{Input: } P_z\text{: Distribution of latent initializations.} \\
\begin{aligned}
1: & \ \mathcal{Z} \sim P_z \quad \quad \quad \quad \quad \quad \quad \quad \quad \quad \, \triangleright \text{Initialize distilled latents} \\
2: & \ \textbf{for each } \text{distillation step\dots} \textbf{ do} \\
3: & \ \quad \mathcal{S} = G(\mathcal{Z}) \quad \quad \quad \quad \quad \quad \quad \quad \ \ \triangleright \text{Get images from latents} \\
4: & \ \quad \mathcal{L} = \text{Alg}(\mathcal{S}, \mathcal{T}) \quad \quad \quad \quad \quad \quad \ \, \triangleright \text{Compute distillation loss} \\
5: & \ \quad \mathcal{Z} \leftarrow \text{SGD}(\mathcal{Z}; \mathcal{L}) \quad \quad \quad \quad \quad \, \triangleright \text{Update } \mathcal{Z} \text{ with respect to } \mathcal{L} \\
6: & \ \textbf{for end}
\end{aligned} \\
\textbf{Output: } \text{Distilled images } \mathcal{S} = G(\mathcal{Z}) \\
\hline
\end{array}
$$

所以 GLaD 的方法实际上非常简单，我们仅仅是在被初始化对象这里加上一个生成模型，可以是 VAE Decoder，原文则使用 StyleGAN-XL generator。这也许是考虑到 VAE Decoder 生成语义先验不够强烈。但是在后续的诸多方法中，生成器的变化可以做大文章，比如我们后续将要提到的 LD3M。

原始算法是
$$\mathcal S
\overset{\mathrm{DC/DM/MTT}}{\longrightarrow}
\mathcal L.$$
现在只是改为
$$\mathcal Z
\overset{G}{\longrightarrow}
\mathcal S
\overset{\mathrm{DC/DM/MTT}}{\longrightarrow}
\mathcal L.$$

GLaD 方法的动机是什么？我们认为，传统像素优化自由度太高。对于一张合成图像
$$\widetilde x
\in
\mathbb R^{H\times W\times3},$$
传统的像素优化方法认为每个像素都可以进行优化，但是这会带来无比巨大的优化自由度。这很容易将蒸馏过程引入局部的投机技巧，比如创造某种形状特殊纹理来应对特殊的蒸馏数据集对应架构。这被我们认为就是泛化性低下的根源，一旦优化过程发现了对于某种架构专属的蒸馏数据优化方法，其不再有动力去寻找更加泛化的优化方案。

所以实际上，GLaD 方法的生成器并不仅仅是一种翻译器，而是类似一种参数优化限制。我们要求算法仅仅允许更新潜空间低维元素，这里的优化自由度远远低于原始像素空间，这通常会带来更加连续的局部纹理与更好的架构泛化性。

但是，GLaD 方法也存在隐患。如果我们的生成器过于强大，会导致原始潜空间被受到拉扯向真实数据流形巨大的吸引力，反而导致寻找最优蒸馏图像的困难。

最后我们来说使用生成器的一个缺点也是最大的缺点，那就是反向传播时会制造巨大的显存负担，这是因为梯度传播时需要穿过整个生成器神经网络。作者提出了一种优化，第一次前向传播时，先计算
$$\mathcal S
=
G(\mathcal Z),$$
这一步先不展开计算图，然后计算蒸馏损失。这一步需要展开计算图
$$\mathcal L
=
\operatorname{Alg}
\left(
\mathcal S,\mathcal T
\right).$$
利用计算图计算梯度
$$g_{\mathcal S}
=
\frac{\partial\mathcal L}{\partial\mathcal S}.$$
得到梯度之后删除计算图。

现在做第二次前向传播。执行
$$\mathcal S'
=
G(\mathcal Z),$$
这一步需要展开计算图。然后直接计算
$$\frac{\partial\mathcal L}{\partial\mathcal Z}
=
\left(
\frac{\partial\mathcal S'}{\partial\mathcal Z}
\right)^{\mathsf T}
g_{\mathcal S}.$$
这是一个 JVP，我们可以优化。现在我们得到梯度，可以做正常梯度下降
$$\mathcal Z
\leftarrow
\mathcal Z
-
\eta_z
\frac{\partial\mathcal L}{\partial\mathcal Z},$$

利用以上方法可以减少非常多峰值显存，代价是重复计算生成器结果。

## 成果与讨论

GLaD 原文使用了 StyleGAN 作为生成器，这是一个很早期的生成模型，其将图像生成分为多个维度量级，从最初始最低维潜空间逐渐向更大的维度进行生成。原文非常详细讨论了 StyleGAN 各个维度层级的生成情况，但是我们不聚焦在这里。原因是后续的方法中我们会逐渐淘汰 StyleGAN。

原文实验指出，GLaD 方法展现出了几乎所有架构上更优于原始像素优化的成绩。

一个极其离奇又可以解释的结果是，生成器可以使用一个随机初始化生成器，而不是非常大费周章的预训练生成器，两者效果非常接近。这是因为 GLaD 方法仅仅将生成器视为一种潜空间到真实像素空间之家梯度传播的桥梁和一种语义正则器，我们并不关心生成器本身训练程度如何，也不关心生成器生成图像的好坏。

总之无论如何，GLaD 第一次提出将数据集蒸馏搬运到潜空间，这实际上为后来的数据集蒸馏方案给出了相当多思路。在后续的方法中，我们甚至可以看到将蒸馏直接搬到 Prompt 空间。

下面我们来说说 LD3M，这是对 GLaD 方法的改进版本。

# LD3M

推荐你读 https://arxiv.org/pdf/2403.03881 Unlocking Dataset Distillation with Diffusion Models 这个方法正式名称是 $\textbf{LD}^{3}\textbf{M}$，我们简记为 LD3M。

## 基本逻辑

我们做一件非常简单的事情，将 GLaD 方法中的生成器改为 Diffusion 模型。

原始 GLaD 是
$$Z
\longrightarrow
G_{\mathrm{GAN}}(Z)
\longrightarrow
X_{\mathrm{syn}}
\longrightarrow
\mathcal L_{\mathrm{DD}}.$$
LD3M 则是
$$Z
\longrightarrow
z_T
\longrightarrow
z_{T-1}
\longrightarrow
\cdots
\longrightarrow
z_0
\longrightarrow
D(z_0)
\longrightarrow
X_{\mathrm{syn}}
\longrightarrow
\mathcal L_{\mathrm{DD}}.$$
但是不仅仅这么简单，因为如果直接这样做，显然会导致反向传播更新合成数据集时穿透非常多步数，引向梯度消失的问题。LD3M 为这里加上了特殊的优化技巧，非常类似 ResNet 的思想，我们后续详谈。

我们开始正式叙述。我们初始拥有潜空间族
$$Z=
\{Z_1,\ldots,Z_m\},$$
这个潜空间族的初始化来自真实数据的 Encoding，是从真实数据集中每一类随机选出图片
$$X_s=
\{x_1,\ldots,x_m\},$$
利用 VAE Encoder 编码得到的
$$Z_j=\mathcal{E}(x_j).$$

更多的，LDM 需要条件嵌入。我们通过一个预训练嵌入网络编码条件
$$c_j=C(y_j),$$
其中 $y_j$ 是第 $j$ 张图类别，$c_j$ 是条件向量。

所以，LD3M 定义最终问题为
$$Z^*,c^*
=
\arg\min_{Z,c}
\mathcal L_{\mathrm{DD}}
\left(
D\left[p_\theta(z_0\mid z_T,c)\right],
\mathcal T
\right).$$
其中 $p_\theta(z_0\mid z_T,c)$ 表示模型接收初始加噪结果与条件向量，最终去噪后的 Decoding 结果。

所以，如何进行这个过程？首先，我们对我们得到的初始潜空间族加噪
$$z_T
\sim
q(z_T\mid Z),$$
这里我们会非常熟悉，换言之就是
$$z_T
=
\sqrt{\overline\alpha_T}Z
+
\sqrt{1-\overline\alpha_T}\epsilon,
\qquad
\epsilon\sim\mathcal N(0,I).$$
其中 $\overline\alpha_T$ 是调度系数，根据条件概率路径的类型进行调整。

现在我们利用潜空间图像生成网络进行去噪
$$z_T
\longrightarrow
z_{T-1}
\longrightarrow
\cdots
\longrightarrow
z_0.$$
最终解码
$$X_{\mathrm{syn}}
=
D(z_0).$$
再交给原始蒸馏算法，如 MTT
$$\mathcal L_{\mathrm{DD}}
=
\operatorname{Alg}
\left(
X_{\mathrm{syn}},
\mathcal T
\right).$$

所以实际上 LD3M 也是一个同样简短的算法。下面我们给出完整算法。

$$
\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{Latent Dataset Distillation with Diffusion Models (LD3M)} \\
\hline
\textbf{Input: } \text{randomly selected collection } X_s, \text{ pre-trained encoder } \mathcal{E}, \text{ pre-trained decoder } \mathcal{D}, \text{ pre-trained} \\
\text{denoiser } \mu_{\theta} \text{ with frozen parameters } \theta, \text{ noise levels } \sigma_t. \\
\begin{aligned}
& \mathcal{Z} = \mathcal{E}(X_s) \\
& \mathbf{z}_T \sim q(\mathbf{z}_T \mid \mathcal{Z}) \\
& \textbf{for } t = T, \dots, 1 \textbf{ do} \\
& \quad \varepsilon_t \sim \mathcal{N}(\mathbf{0}, \mathbf{I}) \\
& \quad \mathbf{z}_{t-1} \leftarrow \left( \left(1 - \frac{t}{T}\right) \cdot \mu_{\theta}(\mathbf{c}, \mathbf{z}_t, \gamma_t) + \frac{t}{T} \cdot \mathbf{z}_T \right) + \sigma_t^2 \varepsilon_t \\
& \textbf{for end} \\
& X_{\text{syn}} \leftarrow \mathcal{D}(\mathbf{z}_0)
\end{aligned} \\
\textbf{Return: } X_{\text{syn}} \\
\hline
\end{array}
$$

## 优化

LD3M 方法绝对继承了 GLaD 方法显存负担过重的缺点。更多的，这个问题甚至进一步恶化了，原因是 Diffusion 模型去噪过程是多步的，这意味着反向传播需要穿过非常多的步数。作者因此提出一种有效的解决办法。

假如我们使用原始反向去噪方式
$$\mu_{\theta}(\mathbf{c}, \mathbf{z}_t, \gamma_t) = \frac{1}{\sqrt{\alpha_t}} \left( \mathbf{z}_t - \frac{1 - \alpha_t}{\sqrt{1 - \gamma_t}} f_{\theta}(\mathbf{c}, \mathbf{z}_t, \gamma_t) \right)$$


$$\mathbf{z}_{t-1} \leftarrow \mu_{\theta}(\mathbf{c}, \mathbf{z}_t, \gamma_t) + \sigma_t^2 \varepsilon_t,$$
此处的 $f_{\theta}$ 指的是预训练的图像生成模型参数。

那么当我们试图优化合成数据集，梯度需要被以这种方式计算
$$\frac{\partial\mathcal L}{\partial Z}
=
\frac{\partial\mathcal L}{\partial z_0}
\frac{\partial z_0}{\partial z_1}
\frac{\partial z_1}{\partial z_2}
\cdots
\frac{\partial z_T}{\partial Z}.$$
当步数极多，会导致
$$\left\|
\frac{\partial\mathcal L}{\partial Z}
\right\|
\longrightarrow
0.$$
或者
$$\left\|
\frac{\partial\mathcal L}{\partial Z}
\right\|
\longrightarrow
\infty.$$
这个问题非常像神经网络的梯度消失和梯度爆炸。我们可以用类似的思想解决这个问题。

我们将标准反向去噪改为
$$z_{t-1}
=
\left(
1-\frac{t}{T}
\right)
\mu_\theta(c,z_t,\gamma_t)
+
\frac{t}{T}z_T
+
\sigma_t^2\epsilon_t.$$
其中 $t\in\{T,T-1,\ldots,1\}$ 是当前去噪步数，$T$ 是总去噪步数，$z_T$ 是最初始带噪潜空间元素。

实际上，这就是为反向传播每一步加上一个从 $z_T$ 出发的捷径，这是一个反向去噪版本的 ResNet。我们创造了一种极短的梯度传播路径
$$\mathcal L_{\mathrm{DD}}
\longrightarrow
z_{t-1}
\longrightarrow
z_T
\longrightarrow
Z.$$

但是，这个反向去噪过程正常吗？答案是，显然否。没有任何反向去噪推理方式会将去噪结果向最混沌的结果拉扯。我们这里的去噪方式不是为了生成更好的图像，而是让蒸馏算法可以顺利地优化合成数据集。

显存怎么办？我们做和 GLaD 一模一样的事情，计算两遍减少峰值显存。我们不赘述。所以这里没有更好的优化方法，LD3M 被迫承受显存压力。

## 成果与讨论

LD3M 的方法总是略好于 GLaD 的成果，这意味着这是一个小有进步但不是突破性的方法。

为什么 LD3M 可以有效？这个问题非常有趣，因为看上去我们只是将 GLaD 中的 GAN 改为 Diffusion 模型，然后发现确实有效。作者做了一个实验，第一组是我们上述的 LD3M，第二组则是去掉 Diffusion 模型，换言之第二组就是 GLaD 方法的 GAN 改为 VAE Decoder。最终结果是，这两组效果均好过原始 GLaD 方法，并且第一组效果略好于第二组。

这个结果说明 Diffusion 模型确实有用，但是也许没有那么突破性的益处。

下面这张图给出 GLaD 方法蒸馏图像与 LD3M 方法区别。

<img src="./assets/LD3M.png" width="900" height="230">

可以看到，GLaD 实际上看起来更加真实。这有点诡异，因为按照我们在 RDED 中所提出的说法，真实性高度相关泛化性。所以我们不禁要问：LD3M 相较 GLaD，到底是进步还是退步？

我个人说一点看法，答案是，不要着急。指标上，LD3M 是一种微小的进步，但是带来了更大的计算负担。但是从历史的角度看，LD3M 为后面的一系列基于 Diffusion 的方法带来了基础。

我们快速看下一个方法，D3M。

# D3M

原文是 https://arxiv.org/pdf/2403.07142 One Category One Prompt: Dataset Distillation using Diffusion Models 原名是 $\text{D}^{3}\text{M}$，我们还是简称 D3M。

## 基本逻辑

LD3M 指出，数据集蒸馏的结果可以来自图像生成模型，只需要我们给出一定的原始加噪图像与合适的 Prompt。那么在大家都可以访问同一个 T2I 基础图像生成模型的前提下，一个类别是否可以直接压缩成一个低维 Prompt 或者 Prompt 的嵌入向量？

以上这个疑问引出了 D3M 核心思想。

我们记真实数据集是
$$\mathcal D
=
\{(x_i,y_i)\}_{i=1}^{N},$$
其中 $x_i\in\mathbb R^{H\times W\times3}$，$y_i\in\{1,\ldots,C\}$ 是类别。

对于每个真实图像 $x_i$，我们裁剪出多个 patch
$$x_{i,1}',x_{i,2}',\ldots,x_{i,K}'.$$
现在我们将每个 patch 输入预训练教师模型 $f$ 计算其对于原始类别 $y_i$ 的交叉熵损失
$$\ell
\left(
f(x_{i,k}'),y_i
\right).$$
现在我们选出最具代表性的 patch，这里的操作和 RDED 几乎一模一样
$$x_i^*
=
\arg\min_{x_{i,k}'}
\ell
\left(
f(x_{i,k}'),y_i
\right).$$

记被挑选出的集合是
$$\mathcal D_c^*
=
\{x_i^*\mid y_i=c\}.$$
随后也做和 RDED 几乎一样的事情，我们将统一类别下的多个 patch 拼为一个 collage
$$X_c^{\mathrm{collage}}
=
\operatorname{Grid}
\left(
x_{c,1}^*,\ldots,x_{c,16}^*
\right).$$
这是第一阶段全部内容。实际上我们在构建类别 $c$ 下的数据分布情况
$$P_{\mathrm{collage}}(X\mid c).$$

在第二阶段，我们需要用到一个预训练的冻结 T2I 潜空间图像生成网络，包括 VAE 与 网络主体。接下来的叙述中我们默认整个潜空间图像生成模型是一体的。对于每个类别 $c$，我们创建一个不在原始图表中的 token $S_c^*$，那么这个 token 会对应一个 embedding $v_c$，这个 embedding 是可学习的。

那么如何让这个 embedding 学习？实际上思想非常简单。当我们为模型输入这个 embedding 与 $X_c^{\mathrm{collage}}$ 的前向加噪图像，我们要求模型预测出原始 $X_c^{\mathrm{collage}}$，这样我们可以保证 $v_c$ 实际上是 $X_c^{\mathrm{collage}}$ 的 textual-inversion。换言之
$$v_c^*
=
\arg\min_{v_c}
\mathbb E_{X_c,t,\epsilon}
\left[
\left\|
\epsilon
-
\epsilon_\theta
\left(
X_c(t),t,\rho(v_c)
\right)
\right\|_2^2
\right].$$
其中 $\rho(v_c)$ 是含有 $v_c$ 文本信息。

现在，如何给出最终的蒸馏数据集？我们拥有训练良好的 $v_c^*$ 与一个随机种子 $r_j$，我们使用生成模型的采样器
$$\widetilde x_{c,j}
=
G_{\mathrm{diff}}
\left(
v_c^*,r_j
\right).$$
其中 $r_j$ 决定了初始的高斯噪声。这意味当 IPC 变大，我们不需要更多的 token，仅仅需要改变更多初始随机种子
$$r_1,r_2,\ldots,r_m
\quad\Longrightarrow\quad
\widetilde x_{c,1},
\widetilde x_{c,2},
\ldots,
\widetilde x_{c,m}.$$

最后，蒸馏数据集如何给出标签？最简单想法仍然是使用 $v_c$ 继承的类别标签 $c$。但是显然的，我们更愿意使用 Soft Label。对于生成图像
$$\widetilde x_{c,j}
=
G_{\mathrm{diff}}
\left(
v_c^*,r_j
\right).$$
我们将其切为多个区域
$$\widetilde x_{c,j}
=
\{
\widetilde x_{c,j,1},
\ldots,
\widetilde x_{c,j,M}
\},$$
再使用教师模型 $f$ 对其打上标签
$$q_{c,j,m}
=
\operatorname{softmax}
\left(
F(\widetilde x_{c,j,m})
\right).$$
学生模型训练时同样对于每个被切割区域求出交叉熵损失再做均值。这里和 RDED 也是一模一样。

因此，蒸馏数据集完全不需要保存图像，而是仅仅 token 与随机种子与 Soft Labels
$$\left(
v_c,r_j,q_{c,j}
\right).$$

下面这两张图展示了完整的流程。

<img src="./assets/D3M1.png" width="900" height="280">
<img src="./assets/D3M2.png" width="900" height="360">

## 成果与讨论

D3M 高度依赖一个假设，那就是大家拥有同一套 T2I 图像生成模型。如果这个假设不成立，那么蒸馏根本无从谈起。

更多的，尽管我们认为 T2I 模型内部已经包含了众多的先验知识，其实一个 Prompt 是否能覆盖整个类别分布还是非常可疑。不同随机种子带来的可能只是外观随机性，而不一定是有效训练模式。相比 D4M 直接做 K-means 承认多峰分布，D3M 有些稚嫩。



一件我认为最反直觉的事情是，图像生成模型天生被用来生成一整张图，但是 D3M 却要求其生成多张图片的机械拼接？这不一定是图像生成模型擅长做的。实际上更自然的做法绝对是让模型一张一张生成之后拼接，但是我们没有这么做。

无论如何，D3M 最终表现还行，可以和 RDED 在性能上进行对抗。更多的，如果我们不统计 T2I 模型参数而是仅仅计算 token 等等内容容量，D3M 的压缩在储存单位上效率很高。

然而一个事实是，D3M 方法随着 IPC 上升饱和速度很快。这意味着实际上更多的随机种子并不会形成更多的类内模式，这方面在 RDED 上比较劣势。

# 总结

本章我们详细探讨了基于生成模型的数据集蒸馏方法，下一章我们继续这些内容。

一个题外话是，这些章节写得很快，至少比我写图像生成主线快得多，原因也许是数据集蒸馏的方法非常易懂。